## Blog Stone Project - Adding Uers

1. install flask

In [ ]:
pip install flask flask-sqlalchemy flask-login flask-wtf

2. import everything

In [ ]:
from flask import (
    Flask,
    render_template,
    request,
    redirect,
    url_for,
    flash
)

from flask_sqlalchemy import SQLAlchemy

from flask_login import (
    UserMixin,
    LoginManager,
    login_user,
    logout_user,
    login_required,
    current_user
)

from werkzeug.security import (
    generate_password_hash,
    check_password_hash
)

3. configure flask

In [ ]:
app = Flask(__name__)

app.config["SECRET_KEY"] = "your-secret-key"

app.config["SQLALCHEMY_DATABASE_URI"] = "sqlite:///blog.db"

app.config["SQLALCHEMY_TRACK_MODIFICATIONS"] = False

db = SQLAlchemy(app)

4. create the user model

In [ ]:
class User(UserMixin, db.Model):

    id = db.Column(
        db.Integer,
        primary_key=True
    )

    email = db.Column(
        db.String(100),
        unique=True,
        nullable=False
    )

    password = db.Column(
        db.String(200),
        nullable=False
    )

    name = db.Column(
        db.String(100),
        nullable=False
    )

5. create the blog post model

In [ ]:
class BlogPost(db.Model):

    id = db.Column(
        db.Integer,
        primary_key=True
    )

    title = db.Column(
        db.String(250),
        unique=True,
        nullable=False
    )

    subtitle = db.Column(
        db.String(250),
        nullable=False
    )

    date = db.Column(
        db.String(250),
        nullable=False
    )

    body = db.Column(
        db.Text,
        nullable=False
    )

    author = db.Column(
        db.String(250),
        nullable=False
    )

    img_url = db.Column(
        db.String(500),
        nullable=False
    )

6. addd the relationship

In [ ]:
class BlogPost(db.Model):

    id = db.Column(
        db.Integer,
        primary_key=True
    )

    title = db.Column(
        db.String(250),
        unique=True,
        nullable=False
    )

    subtitle = db.Column(
        db.String(250),
        nullable=False
    )

    date = db.Column(
        db.String(250),
        nullable=False
    )

    body = db.Column(
        db.Text,
        nullable=False
    )

    img_url = db.Column(
        db.String(500),
        nullable=False
    )

    author_id = db.Column(
        db.Integer,
        db.ForeignKey("user.id"),
        nullable=False
    )

    author = db.relationship(
        "User",
        back_populates="posts"
    )

add to user

In [ ]:
posts = db.relationship(
    "BlogPost",
    back_populates="author"
)

8. Requirement 1 — Register users

In [ ]:
@app.route("/register", methods=["GET", "POST"])
def register():

    if request.method == "POST":

        email = request.form["email"]

        existing_user = db.session.execute(
            db.select(User).where(
                User.email == email
            )
        ).scalar()

        if existing_user:
            flash("You've already registered with this email.")
            return redirect(url_for("login"))

        hashed_password = generate_password_hash(
            request.form["password"],
            method="pbkdf2:sha256",
            salt_length=8
        )

        new_user = User(
            email=email,
            password=hashed_password,
            name=request.form["name"]
        )

        db.session.add(new_user)
        db.session.commit()

        login_user(new_user)

        return redirect(url_for("home"))

    return render_template("register.html")

9. Requirement 2 — Login

In [ ]:
@app.route("/login", methods=["GET", "POST"])
def login():

    if request.method == "POST":

        email = request.form["email"]
        password = request.form["password"]

        user = db.session.execute(
            db.select(User).where(
                User.email == email
            )
        ).scalar()

        if user and check_password_hash(
            user.password,
            password
        ):

            login_user(user)

            return redirect(url_for("home"))

        flash("Invalid email or password.")

    return render_template("login.html")

10. User loader

In [ ]:
@login_manager.user_loader
def load_user(user_id):

    return db.get_or_404(
        User,
        int(user_id)
    )

11. Requirement 3 — Protect routes

In [ ]:
@app.route("/make-post")
def make_post():
    return render_template("make-post.html")

12. logout

In [ ]:
@app.route("/logout")
@login_required
def logout():

    logout_user()

    return redirect(url_for("home"))